### 多工具并行调用

In [25]:
import json
import os

import dotenv
import httpx

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 加载环境变量
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


@tool
def get_weather(loc: str) -> str:
    """
    查询即时天气函数
    :param loc: 必要参数，字符串类型，用于表示查询天气的具体城市名称，\
    注意，中国的城市需要用对应城市的英文名称代替，例如如果需要查询北京市天气，则loc参数需要输入'Beijing'；
    :return：OpenWeather API查询即时天气的结果，具体URL请求地址为：https://api.openweathermap.org/data/2.5/weather\
    返回结果对象类型为解析之后的JSON格式对象，并用字符串形式进行表示，其中包含了全部重要的天气信息
    """
    # Step 1.构建请求
    url = "https://api.openweathermap.org/data/2.5/weather"
    # Step 2.设置查询参数
    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }
    # Step 3.发送GET请求
    response = httpx.get(url, params=params)
    # Step 4.解析响应
    data = response.json()

    return json.dumps(data, ensure_ascii=False)


# 初始化模型（新版推荐）
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=deepseek_api_key
)

# 创建 Agent（最新版推荐）
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="你是天气助手，请根据用户的问题，给出相应的天气信息。"
)

# 调用 Agent
for step in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "请问今天北京和上海天气怎么样，哪个城市更热？"}
        ]
    }
):
    print("\n================ STEP ================")
    print(step)


================ STEP ================
{'model': {'messages': [AIMessage(content='好的，我先查询一下北京和上海的天气情况。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 400, 'total_tokens': 487, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384}, 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 16}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '72179733-b8f6-4c2e-9bd6-0398f579193c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e248f-364e-7230-bae1-3753525d67d0-0', tool_calls=[{'name': 'get_weather', 'args': {'loc': 'Beijing'}, 'id': 'call_00_kiLPkCcIubjRAYEzGVSt8175', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'loc': 'Shanghai'}, 'id': 'call_01_ofPl5uqHtAhIULJJf1u07676', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens':

### 多工具串联调用

In [36]:
import json
import os
import dotenv
import httpx

from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

from langgraph.prebuilt import create_react_agent


# =========================
# 加载环境变量
# =========================
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


# =========================
# 工具1：天气查询
# =========================
@tool
def get_weather(loc):
    """
    查询即时天气函数
    :param loc: 城市英文名（如 Beijing / Shanghai）
    :return: 天气JSON字符串
    """

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": loc,
        "appid": os.getenv("OPENWEATHER_API_KEY"),
        "units": "metric",
        "lang": "zh_cn"
    }

    response = httpx.get(url, params=params)
    data = response.json()

    return json.dumps(data, ensure_ascii=False)


# =========================
# 工具2：写文件
# =========================
@tool
def write_file(content):
    """
    将指定内容写入本地文件
    """

    print(f"写入文件内容：{content}")

    with open("result.txt", "w", encoding="utf-8") as f:
        f.write(content)

    return "已成功写入本地文件。"


# =========================
# 初始化 LLM（DeepSeek在线）
# =========================
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=deepseek_api_key
)


# =========================
# 创建工具列表
# =========================
tools = [get_weather, write_file]


# =========================
# 创建 Agent（关键替换点）
# =========================
agent = create_react_agent(
    model=llm,
    tools=tools,
)


# =========================
# 执行任务
# =========================
result = agent.invoke(
    {
        "messages": [
            ("user", "查一下北京和上海现在的温度，并将结果写入本地的文件中。")
        ]
    }
)


# =========================
# 打印结果（最终答案）
# =========================
print("\n========== 最终结果 ==========\n")
print(result["messages"][-1].content)

C:\Users\zhanghailong\AppData\Local\Temp\ipykernel_9956\4240316208.py:83: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


写入文件内容：北京和上海当前温度查询结果
查询时间：根据数据时间戳换算

【北京】🌤
温度：27.94°C
体感温度：27.48°C
天气：阴，多云
湿度：38%
风速：2.19 m/s

【上海】☁️
温度：27.92°C
体感温度：29.02°C
天气：多云
湿度：57%
风速：6.0 m/s


========== 最终结果 ==========

已完成！以下是查询结果的总结：

---

### 📍 北京
- **温度**：**27.94°C**
- **体感温度**：27.48°C
- **天气**：阴，多云
- **湿度**：38%

### 📍 上海
- **温度**：**27.92°C**
- **体感温度**：29.02°C
- **天气**：多云
- **湿度**：57%

可以看到，北京和上海当前温度非常接近（相差仅0.02°C），但北京的湿度较低，体感更干爽；上海湿度较高，体感温度略高于实际气温。

结果已成功写入本地文件中 📄


## 项目实践
### 联网搜索问答
langchain 内置了常用的第三方搜索库，具体可参考文档：https://python.langchain.com/docs/integrations/tools/#search，常用的 google 搜索访问受限，在这里我们模拟浏览器访问百度搜索页面，完成联网搜索问答。


In [1]:
%pip install langchain langchain-core beautifulsoup4 httpx python-dotenv

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import dotenv
import httpx

from bs4 import BeautifulSoup

from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# =========================
# 加载环境变量
# =========================
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


# =========================
# 百度搜索工具
# =========================
@tool
def baidu_search(query: str) -> str:
    """
    使用百度搜索获取实时信息。

    :param query: 搜索关键词
    :return: 百度搜索结果摘要
    """

    # 构建百度搜索URL
    url = "https://www.baidu.com/s"

    params = {
        "wd": query
    }

    # 请求头（非常重要）
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        )
    }

    # 发送请求
    response = httpx.get(
        url,
        params=params,
        headers=headers,
        timeout=10
    )

    # HTML解析
    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    # 提取搜索结果
    for item in soup.select(".result")[:5]:

        title = item.select_one("h3")

        if title:
            text = title.get_text(strip=True)
            results.append(text)

    # 返回结果
    if results:
        return "\n".join(results)

    return "未搜索到相关内容"


# =========================
# 初始化 DeepSeek 模型
# =========================
llm = init_chat_model(
    "deepseek-chat",
    model_provider="deepseek",
    api_key=deepseek_api_key
)

# =========================
# 创建 Agent
# =========================
agent = create_agent(
    model=llm,
    tools=[baidu_search],
    system_prompt=(
        "你是一个联网搜索助手。"
        "当用户询问实时信息时，请主动调用搜索工具。"
    )
)

# =========================
# 调用 Agent
# =========================
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "小米最近发布了什么新品？"
            }
        ]
    }
)

# =========================
# 输出结果
# =========================
print("\n===== 最终回答 =====\n")

print(result["messages"][-1].content)


===== 最终回答 =====

根据搜索结果，以下是近期小米发布的一些值得关注的新品：

---

### 1️⃣ **REDMI Turbo 4**
- 这是Redmi品牌推出的中端性能机型，主打高性价比和强劲性能。
- 搭载了高性能芯片（具体配置随版本不同有所差异）。

### 2️⃣ **小米MIX Flip 2**（2025年6月26日发布）
- 这是小米旗下的小折叠屏手机，预计将在折叠形态、影像和续航上有所升级。
- 具体配置和售价信息可以关注后续官方公布。

### 3️⃣ **小米15周年战略新品发布**
- 2025年是小米成立15周年，小米举办了一场重要的战略新品发布会，推出了多款旗舰产品。

### 4️⃣ **小米折叠屏收束，高端支线爆发**
- 有消息称小米在2025年调整了产品线策略，折叠屏机型可能有所收束，但高端机型（如Ultra系列等）持续发力。

---

> ⚠️ 由于搜索结果有一定局限性，具体的新品详细参数、售价和上市时间，建议你关注**小米官网**、**小米商城**或**小米官方微博**获取最准确的信息。如果你想了解某一款产品的具体细节，也可以告诉我，我帮你进一步查找！


在大模型应用开发领域有非常多的需求场景，其中一个比较热门的就是浏览器自动化，通过自动化提取网页内容，然后进行分析，最后生成报告。这样的流程提升效率和收集信息的有效途径。因此接下来，我们就尝试使用尝试使用create_openai_tools_agent来实际开发一个浏览器自动化代理。代码如下：

In [3]:
%pip install langchain langchain-community playwright

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/37.7 MB ? eta -:--:--
     --- ------------------------------------ 3.7/37.7 MB 16.8 MB/s eta 0:00:03
     ------ --------------------------------- 6.0/37.7 MB 21.8 MB/s eta 0:00:02
     ----------------- --------------------- 17.0/37.7 MB 26.2 MB/s eta 0:00:01
     ------------------------------- ------- 30.9/37.7 MB 34.5 MB/s eta 0:00:01
     --------------------------------------  37.5/37.7 MB 37.2 MB/s eta 0:00:01
     ---------------------------------------- 37.7/37.7 MB 28.5 MB/s  0:00:01

   ---------------------------------------- 0/2 [pyee]
   ---------------------------------------- 0/2 [pyee]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ------------------- 1/2 [playwright]
   -------------------- ----------

In [ ]:
import os
import asyncio
import dotenv

from playwright.async_api import async_playwright

from langchain_community.agent_toolkits import PlayWrightBrowserToolkit

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# =========================
# 加载环境变量
# =========================
dotenv.load_dotenv()

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


async def main():

    # =========================
    # 启动 Playwright
    # =========================
    playwright = await async_playwright().start()

    # 启动 Chromium 浏览器
    browser = await playwright.chromium.launch(
    channel="chrome",
    headless=False
    )

    # =========================
    # 创建 Toolkit
    # =========================
    toolkit = PlayWrightBrowserToolkit.from_browser(
        async_browser=browser
    )

    # 获取工具
    tools = toolkit.get_tools()

    print("\n===== 已加载工具 =====")

    for tool in tools:
        print(tool.name)

    # =========================
    # 初始化 DeepSeek
    # =========================
    llm = init_chat_model(
        "deepseek-chat",
        model_provider="deepseek",
        api_key=deepseek_api_key
    )

    # =========================
    # 创建 Agent
    # =========================
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=(
            "你是一个网页分析助手。"
            "你可以访问网页并总结网页内容。"
        )
    )

    # =========================
    # 用户输入
    # =========================
    user_input = (
        "访问这个网站："
        "https://langchain-doc.cn/v1/python/langchain/agents.html "
        "并帮我总结网站内容"
    )

    print("\n===== 开始执行 =====\n")

    # =========================
    # Agent调用
    # =========================
    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input
                }
            ]
        }
    )

    # =========================
    # 输出结果
    # =========================
    final_msg = result["messages"][-1]

    print("\n===== AI总结 =====\n")

    print(final_msg.content)

    # =========================
    # 关闭浏览器
    # =========================
    await browser.close()
    await playwright.stop()


# =========================
# 程序入口
# =========================
if __name__ == "__main__":
    asyncio.run(main())

NotImplementedError: 

将Playwright Agent封装成工具函数，并结合LangChain的LCEL串行链，实现一个更加复杂的浏览器自动化代理。代码如下：

In [ ]:
import os
import asyncio
from datetime import datetime

import dotenv

# =========================
# Playwright 原生异步API
# =========================
from playwright.async_api import async_playwright

# =========================
# LangChain Browser Toolkit
# =========================
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit

# =========================
# 2026 推荐统一模型入口
# =========================
from langchain.chat_models import init_chat_model

# =========================
# 2026 推荐 Agent API
# =========================
from langchain.agents import create_agent

# =========================
# LangChain Chain 组件
# =========================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# =========================
# 加载环境变量
# =========================
dotenv.load_dotenv(override=True)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")


# =========================
# 网站总结函数
# =========================
async def summarize_website(url: str) -> str:
    """
    访问指定网站并返回内容总结。

    参数:
        url (str): 要访问和总结的网页URL。

    返回:
        str: 网页正文内容总结。
    """

    try:

        # =========================
        # 启动 Playwright
        # =========================
        playwright = await async_playwright().start()

        # =========================
        # 启动 Chrome 浏览器
        # 使用本地Chrome，避免 playwright install 下载失败
        # =========================
        browser = await playwright.chromium.launch(
            executable_path=r"C:\Program Files\Google\Chrome\Application\chrome.exe",
            headless=False
        )

        # =========================
        # 创建 Browser Toolkit
        # =========================
        toolkit = PlayWrightBrowserToolkit.from_browser(
            async_browser=browser
        )

        # 获取浏览器工具
        tools = toolkit.get_tools()

        print("\n===== 已加载 Browser Tools =====")

        for tool in tools:
            print(tool.name)

        # =========================
        # 初始化 DeepSeek 在线模型
        # =========================
        llm = init_chat_model(
            "deepseek-chat",
            model_provider="deepseek",
            api_key=deepseek_api_key
        )

        # =========================
        # 创建 Browser Agent
        # =========================
        agent = create_agent(
            model=llm,
            tools=tools,
            system_prompt=(
                "你是一个网页分析助手。"
                "你可以自动访问网页、读取网页内容并总结正文。"
                "请忽略评论区、版权信息、友情链接等无关内容。"
            )
        )

        # =========================
        # 构造用户任务
        # =========================
        user_input = (
            f"访问这个网站：{url} "
            f"并帮我详细总结网页正文内容，"
            f"不要总结评论区、版权信息、友情链接等内容。"
        )

        print("\n===== Browser Agent 开始执行 =====\n")

        # =========================
        # Agent执行
        # =========================
        result = await agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_input
                    }
                ]
            }
        )

        # 获取最终结果
        final_msg = result["messages"][-1].content

        print("\n===== 网站总结完成 =====\n")

        # =========================
        # 关闭浏览器
        # =========================
        await browser.close()
        await playwright.stop()

        return final_msg

    except Exception as e:

        return f"网站访问失败: {str(e)}"


# =========================
# 保存 Markdown 文件
# =========================
def save_file(summary: str) -> str:
    """
    将文本内容保存为 md 文件。

    参数:
        summary (str): 需要保存的内容。

    返回:
        str: 保存成功的文件路径。
    """

    # 生成文件名
    filename = f"summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"

    # 写入 Markdown 文件
    with open(filename, "w", encoding="utf-8") as f:

        f.write("# 网页内容总结\n\n")

        f.write(summary)

    return filename


# =========================
# 格式化总结内容
# =========================
async def format_summary(summary: str) -> str:
    """
    对网站总结内容进行 Markdown 美化。

    参数:
        summary (str): 原始总结内容。

    返回:
        str: 格式化后的 Markdown 内容。
    """

    # 初始化格式化模型
    llm = init_chat_model(
        "deepseek-chat",
        model_provider="deepseek",
        api_key=deepseek_api_key
    )

    # Prompt模板
    prompt = ChatPromptTemplate.from_template(
        """
请优化以下网站总结内容，使其更适合 Markdown 文档格式。

要求：
1. 添加标题与小节
2. 使用 Markdown 语法
3. 内容结构清晰
4. 保留核心内容
5. 不要添加虚构信息

原始总结内容：
{summary}

优化后的 Markdown 内容：
"""
    )

    # 输出解析器
    parser = StrOutputParser()

    # 构建Chain
    chain = prompt | llm | parser

    # 执行Chain
    result = await chain.ainvoke(
        {
            "summary": summary
        }
    )

    return result


# =========================
# 主流程
# =========================
async def main():

    # 目标网站
    url = "https://langchain-doc.cn/v1/python/langchain/agents.html"

    print("\n===== 开始网站分析 =====\n")

    # 1️⃣ Browser Agent 总结网站
    summary = await summarize_website(url)

    print("\n===== 原始总结 =====\n")

    print(summary)

    # 2️⃣ LLM优化Markdown格式
    formatted_summary = await format_summary(summary)

    print("\n===== Markdown格式优化完成 =====\n")

    # 3️⃣ 保存文件
    filename = save_file(formatted_summary)

    print(f"\n✅ 文件已保存: {filename}")


# =========================
# 程序入口
# =========================
if __name__ == "__main__":

    asyncio.run(main())

# 网页内容总结

好的，这是根据您的要求优化后的 Markdown 格式文档。

---

# LangChain 智能体 (Agents) 文档总结

> **文档标题**: 智能体 | LangChain 中文文档
> **文档版本**: v1.0 / Python 版

## 一、概述

**智能体 (Agent)** 将大语言模型 (LLM) 与各种**工具**相结合，旨在创建能够推理任务、自主决定使用哪些工具，并通过迭代循环寻求解决方案的系统。

- **`create_agent` 函数**：提供了一个可直接用于生产的智能体实现。
- **工作机制**：LLM 智能体在一个循环中运行，反复调用工具以实现目标，直到满足停止条件（例如，模型输出最终答案或达到预设的迭代次数上限）。
- **底层架构**：`create_agent` 使用 **LangGraph** 构建，这是一个基于图 (Graph) 的运行时，其中节点代表执行步骤，边代表步骤之间的连接。

## 二、核心组件

### 1. 模型 (Model)

智能体支持的模型指定方式有两种：

- **静态模型**：在创建智能体时配置一次，运行过程中不变。
    - **通过字符串标识符**: `create_agent("openai:gpt-5", tools=tools)`
    - **通过模型实例**: 如 `ChatOpenAI` 类的实例，可设置 `temperature`、`max_tokens`、`timeout` 等参数。

- **动态模型**：在运行时根据当前状态和上下文（如对话复杂性）动态选择模型。
    - **实现方式**：使用 `@wrap_model_call` 装饰器创建中间件。例如，简单对话使用 `gpt-4o-mini`，复杂对话切换到 `gpt-4o`。

### 2. 工具 (Tools)

智能体的工具能力超越了简单的模型工具绑定，具体表现为：

- **多工具调用**: 可由单个提示触发序列中的多个工具调用。
- **并行工具调用**: 支持适当的并行执行。
- **动态选择**: 根据前一个工具的结果动态决定下一步使用哪个工具。
- **错误处理**: 内置工具重试逻辑和自定义错误处理。
- **状态持久化**: 在多次工具调用之间保持状态。

- **定义工具**: 使用 `@tool` 装饰器将普通函数定义为一个工具，然后传递给 `create_agent`。
- **工具错误处理**: 使用 `@wrap_tool_call` 装饰器创建中间件，以自定义错误消息并返回给模型。
- **ReAct 循环 (推理+行动)**:
    - 智能体在简短的推理步骤与针对性工具调用之间交替执行。
    - 它将工具的执行结果反馈到后续决策中，直到能给出最终答案。
    - 文档提供了一个“查找最受欢迎的无线耳机并检查库存”的完整示例。

### 3. 系统提示 (System Prompt)

- **静态系统提示**: 通过 `system_prompt` 参数直接传递一个字符串。
- **动态系统提示**: 使用 `@dynamic_prompt` 装饰器创建中间件，根据运行时上下文（例如，用户是“专家”还是“初学者”）动态生成不同的提示。

### 4. 调用 (Invocation)

智能体通过向 State 传递消息序列来调用：

```python
result = agent.invoke({"messages": [{"role": "user", "content": "旧金山天气如何？"}]})
```

同时，支持使用 `stream` 方法进行流式传输，以获取执行过程的中间结果。

## 三、高级概念

### 1. 结构化输出 (Structured Output)

通过 `response_format` 参数配置，支持两种策略：

- **`ToolStrategy`**：使用人工工具调用来生成结构化输出。兼容性好，适用于任何支持工具调用的模型。
- **`ProviderStrategy`**：使用模型提供商的原生结构化输出功能。更可靠，但仅限支持该功能的提供商（如 OpenAI）。

> **注意**: 从 v1.0 开始，必须显式指定 `ToolStrategy` 或 `ProviderStrategy`，不再支持直接传递 Pydantic 模型。

### 2. 记忆 (Memory)

智能体通过消息状态自动维护对话历史，实现短期记忆。自定义状态有两种方式：

- **通过中间件定义状态 (推荐)**：当自定义状态需要被特定中间件钩子和附加工具访问时使用。
- **通过 `state_schema` 定义状态**：作为快捷方式，用于定义仅在工具内部使用的自定义状态。

> **注意**: 自定义状态模式**必须**是 `TypedDict` 类型，不再支持 Pydantic 模型或数据类。

### 3. 流式传输 (Streaming)

使用 `agent.stream()` 方法，并设置 `stream_mode="values"`，可以在智能体执行多步骤任务时，实时获取每个步骤的中间进度信息。

### 4. 中间件 (Middleware)

中间件提供了强大的扩展能力，允许在智能体执行的不同阶段自定义其行为：

- **调用模型前**：处理状态（例如，裁剪消息、注入上下文）。
- **调用模型后**：修改或验证模型响应（例如，设置护栏、内容过滤）。
- **工具执行期间**：处理工具抛出的错误。
- **其他场景**：实现动态模型选择、添加自定义日志、监控或分析功能。

## 总结

本文档是 LangChain v1.0 (Python 版) 智能体 (Agent) 模块的完整指南。它全面介绍了如何使用 `create_agent` 创建生产级别的智能体，内容涵盖了静态/动态模型配置、工具定义与错误处理、系统提示、结构化输出、记忆管理、流式传输以及强大的中间件扩展机制等核心概念与高级用法。